In [1]:
import numpy as np
#%matplotlib notebook
import matplotlib.pyplot as plt
#import sympy as sy
import os
import pandas as pd
from scipy.stats import pearsonr
from scipy.stats import kendalltau
from scipy.stats import spearmanr
from scipy.stats import normaltest
from scipy.stats import linregress
pd.set_option('display.max_rows', 20)
from numpy.polynomial.polynomial import polyfit
plt.rcParams['figure.dpi'] = 100
plt.rcParams['lines.markersize'] = 4
plt.rcParams['font.size'] = 18
plt.rcParams['axes.prop_cycle'] = plt.cycler('color', 'brgmyk')
#plt.rcParams["font.family"] = "Times New Roman"
from matplotlib.ticker import FormatStrFormatter

In [2]:
def load_gas_files(location, 
                   altitudes=[68, 46, 32, 22],
                   base_paths={'n2o': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o',
                               'o3': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_o3'},
                   has_header=False):
    """
    Load N2O and O3 data files for a given location and pressure altitudes.
    
    This function assumes:
      - Files are stored in subdirectories by altitude (e.g., './n2o/68/').
      - Each file is named as "gas_location_pressure.csv", for example, 
        "n2o_quistococha_68.csv".
      - Each file contains two columns:
            1. A date column (formatted like 2018311.0)
            2. A gas concentration column.
      - The columns are separated by one or more whitespace characters.
      - If the file does not have a header row, it assigns column names "date" and "concentration".
    
    Parameters:
        location (str): The location name (e.g., 'quistococha').
        altitudes (list of int, optional): List of pressure altitudes to load. Defaults to [68, 46, 32, 22].
        base_paths (dict, optional): Dictionary mapping gas names to their base directories.
                                     Defaults to {'n2o': './n2o', 'o3': './o3'}.
        has_header (bool, optional): Whether the CSV files include a header row. Defaults to False.
        
    Returns:
        dict: A nested dictionary where the first-level keys are the gas names and the
              second-level keys are the pressure altitudes. Each leaf is a pandas DataFrame.
              For example, result['n2o'][68] contains the DataFrame for 'n2o_quistococha_68.csv'.
    """
    data = {}
    
    for gas, folder in base_paths.items():
        data[gas] = {}
        for pressure in altitudes:
            filename = f"{gas}avg_{location}_{pressure}.csv"
            # Construct the path assuming files are in a subfolder named by the pressure altitude.
            file_path = os.path.join(folder, str(pressure), filename)
            try:
                df = pd.read_csv(file_path, sep=',', header=0 if has_header else None,
                                 names=["date", "concentration"], usecols=[0, 1])
                data[gas][pressure] = df
            except FileNotFoundError:
                print(f"File not found: {file_path}")
                data[gas][pressure] = None
                
    return data

def combine_all(gas_data):
    """
    For each altitude in the gas_data dictionary, combine the N2O and O3 data by performing
    an inner join on the 'date' column. The function assumes that each DataFrame has at least
    the columns 'date' and 'concentration'.
    
    Parameters:
        gas_data (dict): A dictionary with keys 'n2o' and 'o3'. Each of these is a dictionary 
                         keyed by altitude, where the value is a DataFrame.
                         
    Returns:
        dict: A dictionary keyed by altitude. Each value is a combined DataFrame with columns:
              'date', 'n2o', and 'o3'.
    """
    combined_data = {}
    
    # Loop over altitudes (assuming both gases have the same altitude keys)
    for altitude in gas_data['n2o']:
        df_n2o = gas_data['n2o'][altitude]
        df_o3  = gas_data['o3'][altitude]
        
        if df_n2o is None or df_o3 is None:
            print(f"Skipping altitude {altitude} hPa because one of the data sets is missing.")
            combined_data[altitude] = None
            continue
        
        # Check that both DataFrames have a 'date' column
        if 'date' not in df_n2o.columns:
            print(f"Altitude {altitude} hPa: 'date' column missing in N2O data. Columns found: {df_n2o.columns}")
            combined_data[altitude] = None
            continue
        if 'date' not in df_o3.columns:
            print(f"Altitude {altitude} hPa: 'date' column missing in O3 data. Columns found: {df_o3.columns}")
            combined_data[altitude] = None
            continue
        
        # Merge on 'date' (inner join keeps only rows with shared dates)
        combined_df = pd.merge(df_n2o, df_o3, on='date', suffixes=('_n2o', '_o3'))
        # Rename columns to be clear
        combined_df.rename(columns={'concentration_n2o': 'n2o',
                                    'concentration_o3': 'o3'}, inplace=True)
        
        combined_data[altitude] = combined_df
        
    return combined_data

In [3]:
def decimal_year(d):
    """MLS dates look like 2018311.0 (YYYYDDD)."""
    d = int(d)
    return d // 1000 + (d % 1000 - 0.5) / 365.25

def detrend_keep_mean(t, y):
    """Remove a least-squares linear trend in time; keep the series mean."""
    slope, intercept = np.polyfit(t, y, 1)
    return y - (slope * t + intercept) + y.mean()

def combine_all(gas_data, detrend=True):
    """Inner-join N2O and O3 on date for each altitude; optionally detrend both gases."""
    combined_data = {}
    for altitude in gas_data['n2o']:
        df_n2o, df_o3 = gas_data['n2o'][altitude], gas_data['o3'][altitude]
        if df_n2o is None or df_o3 is None:
            print(f"Skipping altitude {altitude} hPa because one of the data sets is missing.")
            combined_data[altitude] = None
            continue
        df = (pd.merge(df_n2o, df_o3, on='date', suffixes=('_n2o', '_o3'))
                .rename(columns={'concentration_n2o': 'n2o', 'concentration_o3': 'o3'})
                .dropna(subset=['n2o', 'o3']))
        if detrend and len(df) > 2:
            t = df['date'].map(decimal_year).to_numpy()
            for gas in ('n2o', 'o3'):
                df[gas] = detrend_keep_mean(t, df[gas].to_numpy())
        combined_data[altitude] = df
    return combined_data

In [4]:
def calculate_correlations(locations, 
                           altitudes=[68, 46, 32, 22],
                           base_paths={'n2o': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o',
                                       'o3': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_o3'},
                           has_header=False):
    """
    Calculate and print the correlation coefficients between N₂O and O₃ 
    for a list of locations. For each location, data is loaded from CSV files,
    combined so that only days with measurements for both gases are retained, 
    and the correlation coefficient is calculated for each altitude.

    Parameters:
      - locations (list of str): List of location names used in the file names 
                                 (e.g., ['quistococha', 'another_location']).
      - altitudes (list of int, optional): List of pressure altitudes (default: [68, 46, 32, 22]).
      - base_paths (dict, optional): Dictionary mapping gas names to their base directories.
                                     Defaults to {'n2o': './n2o', 'o3': './o3'}.
      - has_header (bool, optional): Whether the CSV files include a header row (default: False).

    Returns:
      dict: A nested dictionary with locations as the top-level keys and inner dictionaries
            that have altitudes as keys and the corresponding correlation coefficients as values.
            If the correlation cannot be computed (due to missing or insufficient data), the value is None.
    """
    correlations = {}
    
    for loc in locations:
        # Load the data for the given location.
        gas_data = load_gas_files(loc, altitudes=altitudes, base_paths=base_paths, has_header=has_header)
        # Combine the data so that only days with both measurements remain.
        combined_data = combine_all(gas_data)
        
        correlations[loc] = {}
        for altitude in altitudes:
            df = combined_data.get(altitude)
            if df is not None and len(df) > 1:
                # Calculate the Pearson correlation between N₂O and O₃.
                corr_val = df['n2o'].corr(df['o3'])
                correlations[loc][altitude] = corr_val
            else:
                correlations[loc][altitude] = None
                
    return correlations


In [5]:
# Define a mapping from internal location names to reader-friendly names.
locations_dict = {
    'bashkortostan': 'Bashkortostan',
    'bozeman': 'Montana',
    'brunei': 'Kalimantan',
    'california': 'California',
    'catalonia': 'Catalonia',
    'colombia': 'Colombia',
    'estonia': 'Estonia',
    'finland': 'Finland',
    'florianopolis': 'Santa Catarina',
    'florida': 'Florida',
    'france': 'France',
    'french_guiana': 'French Guiana',
    'huntingdon': 'Southern Québec',
    'iceland_e': 'Iceland (E)',
    'iceland_w': 'Iceland (W)',
    'khabarovsk': 'Khabarovsk',
    'kongo': 'Congo',
    'kyrgyzstan': 'Kyrgyzstan',
    'mexico': 'Xochimilco, Mexico City',
    'morocco': 'Morocco',
    'mukhrino': 'Mukhrino',
    'myanmar': 'Myanmar',
    'nz_n': 'New Zealand (N)',
    'nz_s': 'New Zealand (S)',
    'pantanal': 'Pantanal',
    'quistococha': 'Peruvian Amazon',
    'romania': 'Romania',
    'taiwan': 'Taiwan',
    'tarapoto': 'Tarapoto',
    'tasmania': 'Tasmania',
    'tierra_del_fuego': 'Tierra del Fuego',
    'uganda_e': 'Uganda (E)',
    'uganda_n': 'Uganda (N)',
    'uganda_s': 'Uganda (S)',
    'wales': 'Wales'
    # Add more mappings as needed.
}

In [6]:
def calculate_correlations_with_p(locations, 
                                  altitudes=[68, 46, 32, 22],
                                  base_paths={'n2o': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o',
                                              'o3': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_o3'},
                                  has_header=False):
    """
    For each location, load the gas data, combine it so that only days with both N₂O and O₃ measurements remain,
    and then calculate the Pearson correlation coefficient and p-value for each altitude using scipy.stats.pearsonr.
    
    Parameters:
      - locations (list of str): List of location names used in the file names.
      - altitudes (list of int, optional): List of pressure altitudes (default: [68, 46, 32, 22]).
      - base_paths (dict, optional): Dictionary mapping gas names to their base directories.
      - has_header (bool, optional): Whether the CSV files include a header row.
      
    Returns:
      dict: A nested dictionary with top-level keys as locations. For each location, the inner dictionary
            has altitudes as keys and a dictionary with keys 'r' (the correlation coefficient) and 'p' (the p-value)
            as values. If the correlation cannot be computed, both are set to None.
    """
    correlations = {}
    
    for loc in locations:
        # Load the data for the given location.
        gas_data = load_gas_files(loc, altitudes=altitudes, base_paths=base_paths, has_header=has_header)
        # Combine the data to retain only rows with both measurements.
        combined_data = combine_all(gas_data)
        
        correlations[loc] = {}
        for altitude in altitudes:
            df = combined_data.get(altitude)
            if df is not None and len(df) > 1:
                try:
                    # Compute Pearson correlation and p-value.
                    r, p = pearsonr(df['n2o'], df['o3'])
                    correlations[loc][altitude] = {'r': r**2, 'p': p}
                except Exception as e:
                    correlations[loc][altitude] = {'r': None, 'p': None}
            else:
                correlations[loc][altitude] = {'r': None, 'p': None}
                
    return correlations



In [7]:
def remap_keys(data: dict, key_map: dict) -> dict:
    """Map each key in `data` to key_map[key] when present."""
    return {key_map[k]: v for k, v in data.items() if k in key_map}


#print(remap_keys(corr_results, locations_dict))  

In [8]:
import csv

def save_corr_table(data: dict, path: str, p_tiny_threshold: float = 1e-6):
    """
    data: {
      location: {
        altitude: {"r": <float>, "p": <float>},
        ...
      },
      ...
    }
    Writes CSV with rows:
      location, alt1_r, alt1_p, alt2_r, alt2_p, ...
    No headers. Altitudes are all unique altitudes across locations, sorted.
    Missing (location, altitude) cells are left blank.
    """

    def fmt_r(x):
        # stable, compact numeric formatting for r
        return f"{x:.6g}" if x is not None else ""

    def fmt_p(p):
        if p is None:
            return ""
        if p < p_tiny_threshold:
            return "<<0.001"
        # compact scientific/decimal depending on magnitude
        return f"{p:.6g}"

    # Collect and sort the union of all altitudes
    all_alts = sorted({alt for loc in data.values() for alt in loc.keys()})

    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        for location in sorted(data.keys()):
            row = [location]
            loc_block = data[location]
            for alt in all_alts:
                cell = loc_block.get(alt)
                if cell is None:
                    row.extend(["", ""])        # r, p placeholders
                else:
                    row.extend([fmt_r(cell.get("r")), fmt_p(cell.get("p"))])
            w.writerow(row)

In [9]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

def calculate_seasonal_trends_for_location(
    location,
    altitudes=(68, 46, 32, 22),
    base_paths=None,
    has_header=False
):
    """
    Calculate seasonal N2O–O3 relationships for a single location.
    
    For each altitude and each season (DJF, MAM, JJA, SON), this function:
      - selects only data in that season,
      - fits a simple linear regression O3 ~ N2O,
      - returns r^2 (from Pearson r), p-value, slope, intercept, and n.
    
    Parameters
    ----------
    location : str
        Location identifier used in file names (e.g. 'huntingdon').
    altitudes : iterable of int
        Pressure levels in hPa (e.g. (68, 46, 32, 22)).
    base_paths : dict or None
        Mapping {'n2o': path, 'o3': path}. If None, uses the defaults from your notebook.
    has_header : bool
        Whether the CSV files have a header row.
    
    Returns
    -------
    dict
        Nested dictionary of the form:
          {
            altitude1: {
                'DJF': {'r2': ..., 'p': ..., 'slope': ..., 'intercept': ..., 'n': ...},
                'MAM': {...},
                'JJA': {...},
                'SON': {...}
            },
            altitude2: {
                ...
            },
            ...
          }
        If a season has too few points (< 3) or data is missing, its value is None.
    """
    if base_paths is None:
        base_paths = {
            'n2o': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_n2o',
            'o3': r'C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\correlations\data_o3'
        }

    # Load and combine gas data using your existing helpers
    gas_data = load_gas_files(
        location=location,
        altitudes=list(altitudes),
        base_paths=base_paths,
        has_header=has_header
    )
    combined_data = combine_all(gas_data)

    # Season mapping
    season_map = {
        12: 'DJF', 1: 'DJF', 2: 'DJF',
        3: 'MAM', 4: 'MAM', 5: 'MAM',
        6: 'JJA', 7: 'JJA', 8: 'JJA',
        9: 'SON', 10: 'SON', 11: 'SON'
    }
    season_order = ['DJF', 'MAM', 'JJA', 'SON']

    results = {}

    for alt in altitudes:
        df = combined_data.get(alt)
        alt_result = {}

        if df is None or df.empty or not {'date', 'n2o', 'o3'}.issubset(df.columns):
            # No usable data at this altitude
            for season in season_order:
                alt_result[season] = None
            results[alt] = alt_result
            continue

        # Parse date (yyyy + day-of-year like 2018311.0) -> month
        date_int = df['date'].astype(int)
        date_str = date_int.astype(str)
        dates = pd.to_datetime(date_str, format='%Y%j', errors='coerce')
        months = dates.dt.month

        df = df.copy()
        df['month'] = months
        df['season'] = df['month'].map(season_map)

        for season in season_order:
            subset = df[df['season'] == season][['n2o', 'o3']].dropna()

            if len(subset) < 3:
                alt_result[season] = None
                continue

            x = subset['n2o'].values
            y = subset['o3'].values

            # Pearson correlation + p-value
            try:
                r, p = pearsonr(x, y)
                r2 = r ** 2
            except Exception:
                alt_result[season] = None
                continue

            # Simple linear fit O3 ~ N2O
            try:
                slope, intercept = np.polyfit(x, y, 1)
            except Exception:
                slope, intercept = None, None

            alt_result[season] = {
                'r2': r2,
                'p': p,
                'slope': slope,
                #'intercept': intercept,
                'n': len(subset)
            }

        results[alt] = alt_result

    return results


In [10]:
def seasonal_results_to_df(seasonal_results):
    rows = []
    for alt, seasons in seasonal_results.items():
        for season, vals in seasons.items():
            if vals is None:
                rows.append({
                    'altitude': alt,
                    'season': season,
                    'r2': None,
                    'p': None,
                    'slope': None,
                    #'intercept': None,
                    'n': 0
                })
            else:
                row = {'altitude': alt, 'season': season}
                row.update(vals)
                rows.append(row)
    return pd.DataFrame(rows)


In [11]:
seasonal_huntingdon = calculate_seasonal_trends_for_location(
    location='huntingdon',
    altitudes=[68, 46, 32, 22],
    has_header=False
)
seasonal_quistococha = calculate_seasonal_trends_for_location(
    location='quistococha',
    altitudes=[68, 46, 32, 22],
    has_header=False
)
seasonal_tdf = calculate_seasonal_trends_for_location(
    location='tierra_del_fuego',
    altitudes=[68, 46, 32, 22],
    has_header=False
)

In [12]:
d = load_gas_files('huntingdon', altitudes=[68])
print(d['n2o'][68].head() if d['n2o'][68] is not None else "not loaded")

      date  concentration
0  2005004     304.197272
1  2005006     309.556413
2  2005013     294.731279
3  2005015     244.820228
4  2005020     240.697176


In [13]:

df_huntingdon = seasonal_results_to_df(seasonal_huntingdon)

df_quistococha = seasonal_results_to_df(seasonal_quistococha)

df_tdf = seasonal_results_to_df(seasonal_tdf)


In [14]:
df_huntingdon

,altitude,season,r2,p,slope,n
0,68,DJF,0.557941,6.991711e-78,-0.014483,430
1,68,MAM,0.520620,1.706302e-70,-0.012515,431
2,68,JJA,0.156084,1.934893e-17,-0.003788,428
3,68,SON,0.514925,3.106757e-69,-0.008921,430
4,46,DJF,0.331510,2.477294e-39,-0.011355,430
5,46,MAM,0.205984,2.742427e-23,-0.007667,431
6,46,JJA,0.001353,4.477879e-01,-0.000343,428
7,46,SON,0.081541,1.638094e-09,-0.003185,430
8,32,DJF,0.000291,7.244935e-01,-0.000212,430
9,32,MAM,0.031698,2.030371e-04,0.001911,431


In [15]:
df_quistococha

,altitude,season,r2,p,slope,n
0,68,DJF,0.005391,1.559055e-01,-0.000390,375
1,68,MAM,0.007664,8.750899e-02,-0.000407,382
2,68,JJA,0.006387,1.019364e-01,-0.000535,420
3,68,SON,0.000949,5.383897e-01,0.000163,401
4,46,DJF,0.130218,5.637949e-13,-0.006372,375
5,46,MAM,0.057258,2.241098e-06,-0.004013,382
6,46,JJA,0.071879,2.397413e-08,-0.005404,420
7,46,SON,0.095238,2.695304e-10,-0.004702,401
8,32,DJF,0.106875,8.655133e-11,-0.009509,375
9,32,MAM,0.089273,2.565866e-09,-0.009105,382


In [16]:
df_tdf

,altitude,season,r2,p,slope,n
0,68,DJF,0.003815,2.340489e-01,0.000481,373
1,68,MAM,0.583798,3.131363e-72,-0.009946,371
2,68,JJA,0.471622,3.551828e-54,-0.009134,379
3,68,SON,0.039661,9.086734e-05,0.002323,381
4,46,DJF,0.074100,9.251173e-08,0.002628,373
5,46,MAM,0.248762,9.947906e-25,-0.005087,371
6,46,JJA,0.146992,1.018788e-14,-0.005911,379
7,46,SON,0.313696,7.612615e-33,0.007338,381
8,32,DJF,0.081602,1.954688e-08,0.006230,373
9,32,MAM,0.136675,1.850663e-13,0.005355,371


In [17]:
df_huntingdon.to_excel('seasonal_corr_huntingdon.xlsx',index=False)
df_quistococha.to_excel('seasonal_corr_quistococha.xlsx',index=False)
df_tdf.to_excel('seasonal_corr_tdf.xlsx',index=False)

In [18]:
seasonal_khabarovsk = calculate_seasonal_trends_for_location(
    location='khabarovsk',
    altitudes=[68, 46, 32, 22],
    has_header=False
)
seasonal_brunei = calculate_seasonal_trends_for_location(
    location='brunei',
    altitudes=[68, 46, 32, 22],
    has_header=False
)
seasonal_tasmania = calculate_seasonal_trends_for_location(
    location='tasmania',
    altitudes=[68, 46, 32, 22],
    has_header=False
)

In [19]:

df_khabarovsk = seasonal_results_to_df(seasonal_khabarovsk)

df_brunei = seasonal_results_to_df(seasonal_brunei)

df_tasmania = seasonal_results_to_df(seasonal_tasmania)


In [20]:
df_khabarovsk

,altitude,season,r2,p,slope,n
0,68,DJF,0.420353,2.156354e-57,-0.008273,470
1,68,MAM,0.286264,4.338741e-36,-0.007273,469
2,68,JJA,0.373166,4.153556e-49,-0.007106,467
3,68,SON,0.698047,2.684799e-124,-0.009780,472
4,46,DJF,0.054933,2.755878e-07,-0.002795,470
5,46,MAM,0.003324,2.126925e-01,-0.000651,469
6,46,JJA,0.002687,2.636271e-01,-0.000886,467
7,46,SON,0.115892,2.854841e-14,-0.004556,472
8,32,DJF,0.067598,1.063635e-08,0.002762,470
9,32,MAM,0.227579,5.004427e-28,0.004940,469


In [21]:
df_brunei

,altitude,season,r2,p,slope,n
0,68,DJF,0.032838,5.603071e-04,-0.001188,359
1,68,MAM,0.045294,4.355528e-05,-0.001433,363
2,68,JJA,0.060669,2.316931e-06,-0.001994,359
3,68,SON,0.022939,4.514563e-03,-0.001040,350
4,46,DJF,0.135587,5.708848e-13,-0.006811,359
5,46,MAM,0.063678,1.114522e-06,-0.004798,363
6,46,JJA,0.155298,8.709531e-15,-0.007511,359
7,46,SON,0.208482,1.990337e-19,-0.008238,350
8,32,DJF,0.120178,1.417014e-11,-0.010603,359
9,32,MAM,0.122885,6.189536e-12,-0.011849,363


In [22]:
df_tasmania

,altitude,season,r2,p,slope,n
0,68,DJF,0.226369,4.132365e-17,-0.004943,278
1,68,MAM,0.479849,8.916129e-41,-0.009122,276
2,68,JJA,0.497660,3.287861e-44,-0.012630,285
3,68,SON,0.411738,1.394725e-34,-0.010913,286
4,46,DJF,0.019941,1.848868e-02,-0.001590,278
5,46,MAM,0.168104,1.293742e-12,-0.005296,276
6,46,JJA,0.420359,2.232437e-35,-0.013717,285
7,46,SON,0.147351,1.785192e-11,-0.007165,286
8,32,DJF,0.084548,8.086215e-07,0.004422,278
9,32,MAM,0.163763,2.672820e-12,0.005945,276


In [23]:
df_khabarovsk.to_excel('seasonal_corr_khabarovsk.xlsx',index=False)
df_brunei.to_excel('seasonal_corr_brunei.xlsx',index=False)
df_tasmania.to_excel('seasonal_corr_tasmania.xlsx',index=False)